<a href="https://colab.research.google.com/github/CienciaDatosUdea/002_EstudiantesAprendizajeEstadistico/blob/main/semestre2026-2/Sesiones/Sesion_02b_overfitting_underfitting_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 02b — Subajuste, sobreajuste, validación cruzada y bootstrap

**Fuente teórica:** B. Ghojogh & M. Crowley, *The Theory Behind Overfitting, Cross Validation,
Regularization, Bagging, and Boosting*, arXiv:1905.12787.




## Contenido

| Parte | Tema | Celdas |
|---|---|---|
| 0 | Teoría: sesgo, varianza, ECM, err vs Err | markdown |
| 1 | Subajuste y sobreajuste en vivo | código |
| 2 | Validación cruzada de $K$ particiones | código |
| 3 | Población, muestra y bootstrap | código |
| 4 | Ejercicios propuestos | mixto |


# Parte 0 — Medidas de Error y Validación Cruzada

## Medidas para una Variable Aleatoria

Sea $X$ el valor verdadero de una población (fijo, no aleatorio) y $\hat{X}$ su estimador, que
**sí** es aleatorio porque depende de la muestra que nos tocó.

La **varianza** mide la dispersión del estimador alrededor de **su propia media** — no alrededor
del valor verdadero:

$$\mathbb{V}\text{ar}(\hat{X}) := \mathbb{E}\!\left[(\hat{X} - \mathbb{E}(\hat{X}))^2\right]
= \mathbb{E}(\hat{X}^2) - \left(\mathbb{E}(\hat{X})\right)^2$$

El **sesgo** mide cuánto se desvía la media del estimador del valor verdadero $X$:

$$\mathbb{S}\text{esgo}(\hat{X}) := \mathbb{E}(\hat{X}) - X$$

Dado que $X$ es una constante fija y $\mathbb{E}(\hat{X})$ es un escalar, el sesgo es en sí mismo
un escalar.

El **Error Cuadrático Medio (ECM)** sí mide la desviación respecto al valor verdadero:

$$\text{ECM}(\hat{X}) := \mathbb{E}\!\left[(\hat{X} - X)^2\right]$$

Sumando y restando $\mathbb{E}(\hat X)$ dentro del cuadrado, el término cruzado se anula y queda la
**descomposición sesgo–varianza**:

$$\boxed{\;\text{ECM}(\hat{X}) = \mathbb{V}\text{ar}(\hat{X}) + \left(\mathbb{S}\text{esgo}(\hat{X})\right)^2\;}$$

> **Ejercicio 0.1 .** La figura muestra cuatro estimadores que buscan el valor verdadero
> $X=0$. Clasifiquen cada uno como sesgo alto/bajo y varianza alta/baja.
>
> **Ejercicio 0.2 .** Si solo pudieran eliminar **uno** de los dos errores, ¿cuál
> eliminarían? Argumenten.

![Ilustracion de varianza y sesgo en el error del estimador](https://github.com/CienciaDatosUdea/002_EstudiantesAprendizajeEstadistico/blob/main/semestre2026-2/Sesiones/imagenes/dart.png?raw=1)


## Medidas para un Modelo

Las etiquetas verdaderas $f_i = f(\boldsymbol{x}_i)$ nunca se observan directamente: en datos
reales vienen corrompidas por ruido, que modelamos como ruido gaussiano aditivo:

$$y_i = f_i + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0,\sigma^2)$$

El ruido satisface en particular:

$$\mathbb{E}(\varepsilon_i) = 0, \qquad \mathbb{E}(\varepsilon_i^2) = \mathbb{V}\text{ar}(\varepsilon_i) = \sigma^2$$

y dado que $f_i$ es una constante fija no aleatoria, $\mathbb{E}(f_i) = f_i$.

Si entrenamos un modelo $\hat{f}$ con los datos $\{(\boldsymbol{x}_i, y_i)\}_{i=1}^n$ y denotamos su
predicción para la instancia $i$ por $\hat{f}_i$, las definiciones de sesgo, varianza y ECM se
aplican directamente a $\hat{f}_i$ como estimador de $f_i$, y la descomposición se cumple para cada $i$.

### El error irreducible

Hay una diferencia crucial entre predecir $f_0$ (la verdad) y predecir $y_0$ (lo que medimos).
Si $(x_0,y_0)$ no se usó para entrenar, entonces $\hat f_0 \perp\!\!\!\perp \varepsilon_0$ y:

$$\mathbb{E}\!\left[(y_0 - \hat f_0)^2\right]
= \underbrace{\left(\mathbb{S}\text{esgo}(\hat f_0)\right)^2}_{\text{modelo demasiado rígido}}
+ \underbrace{\mathbb{V}\text{ar}(\hat f_0)}_{\text{modelo demasiado sensible}}
+ \underbrace{\sigma^2}_{\text{error irreducible}}$$

El tercer término **no depende del modelo**. Ningún algoritmo, por sofisticado que sea, puede
bajar de $\sigma^2$. Si su error de test se estanca en un valor, lo primero que hay que preguntarse
no es qué modelo probar sino **cuánto ruido tienen sus mediciones**.

> **Pregunta para discutir.** En un experimento de laboratorio, ¿qué es $\sigma^2$ físicamente?
> ¿Se puede reducir? ¿Cómo cambia eso la estrategia de modelado?


## ECM para una Instancia **Fuera** del Conjunto de Entrenamiento

Partimos del ECM de $\hat{f}_0$ como estimador de $f_0$ y expandimos usando $y_0 = f_0 + \varepsilon_0$.
Si $(x_0, y_0)$ **no** fue usada para entrenar, entonces $\hat{f}_0 \perp\!\!\!\perp y_0$, el término
cruzado se anula, y agregando sobre las $m$ instancias fuera de la muestra:

$$\mathbb{E}\!\left[\sum_{i=1}^{m}(\hat{f}_i - y_i)^2\right] = \sum_{i=1}^{m}(\hat{f}_i - f_i)^2 + m\sigma^2$$

Denotando el error empírico $\text{err} = \sum(\hat{f}_i - y_i)^2$ y el error verdadero
$\text{Err} = \sum(\hat{f}_i - f_i)^2$:

$$\text{Err} = \mathbb{E}[\text{err}] - m\sigma^2$$

Como $m\sigma^2$ es una constante que no depende del modelo, **minimizar err minimiza correctamente Err**.

> *Nota de rigor:* estas son igualdades **en esperanza**, no identidades exactas para una muestra
> particular. Con $m$ finito, err es un estimador ruidoso de $\text{Err} + m\sigma^2$.

## ECM para una Instancia **Dentro** del Conjunto de Entrenamiento

Ahora $(x_0,y_0)$ sí se usó para entrenar, así que $\hat{f}_0$ e $y_0$ **ya no son independientes**:
el modelo ha visto ese punto y se ha acomodado a su ruido. Aplicando el Estimador de Riesgo
Insesgado de Stein (SURE) al término cruzado y agregando sobre las $n$ instancias:

$$\mathbb{E}\!\left[\sum_{i=1}^{n}(\hat{f}_i - y_i)^2\right] = \sum_{i=1}^{n}(\hat{f}_i - f_i)^2 + n\sigma^2 - 2\sigma^2\sum_{i=1}^{n}\frac{\partial \hat{f}_i}{\partial y_i}$$

y por lo tanto:

$$\text{Err} = \mathbb{E}[\text{err}] - n\sigma^2 + 2\sigma^2\sum_{i=1}^{n}\frac{\partial \hat{f}_i}{\partial y_i}$$

El término $2\sigma^2\sum_i \partial\hat{f}_i/\partial y_i$ mide la **complejidad efectiva** del modelo:
cuantifica cuán sensibles son las predicciones ante perturbaciones de las observaciones individuales
de entrenamiento. Para regresión lineal con $p$ parámetros vale exactamente
$\sum_i \partial\hat f_i/\partial y_i = p$ (lo verificamos numéricamente en el Ejercicio 4.1).

A medida que avanza el entrenamiento, $\text{err}$ disminuye, pero el término de complejidad crece,
lo que eventualmente hace que $\text{Err}$ aumente: **ésta es la caracterización formal del sobreajuste**.

### La asimetría fundamental

| | ¿err refleja Err? |
|---|---|
| Fuera de muestra | Sí, salvo una constante $m\sigma^2$ |
| Dentro de muestra | **No**: lo subestima, y la subestimación *crece* con la complejidad |

Esta asimetría es la justificación teórica de la validación cruzada.

![Modelo de overfitting: (a) Error empirico y error verdadero (b) Ilustracion de la ecuacion anterior](https://github.com/CienciaDatosUdea/002_EstudiantesAprendizajeEstadistico/blob/main/semestre2026-1/Sesiones/imagenes/overfitting_curve.png?raw=1)


## Validación Cruzada de $K$ Particiones

La validación cruzada divide $\mathcal{D}$ en $K$ particiones aproximadamente iguales, disjuntas y
exhaustivas:

$$|\mathcal{D}_1| \approx \cdots \approx |\mathcal{D}_K|, \qquad
\bigcup_{i=1}^{K} \mathcal{D}_i = \mathcal{D}, \qquad
\mathcal{D}_i \cap \mathcal{D}_j = \varnothing \;\; \forall\, i \neq j$$

**Algoritmo 1: Validación Cruzada de $K$ Particiones**

> **Entrada:** conjunto de datos $\mathcal{D}$, número de particiones $K$
> **Salida:** error estimado $\text{Err}$
>
> 1. Dividir $\mathcal{D}$ aleatoriamente en $K$ particiones de tamaño similar.
> 2. **Para** $k = 1, \ldots, K$ **hacer:**
>    - $\mathcal{R} \leftarrow$ partición $k$ de $\mathcal{D}$
>    - $\mathcal{T} \leftarrow \mathcal{D} \setminus \mathcal{R}$
>    - Entrenar el modelo usando $\mathcal{T}$
>    - $\text{Err}_k \leftarrow$ error del modelo entrenado sobre $\mathcal{R}$
> 3. **Fin para**
> 4. $\text{Err} \leftarrow \dfrac{1}{K}\displaystyle\sum_{k=1}^{K} \text{Err}_k$

### Por qué funciona

En los datos de **entrenamiento** el término de complejidad hace que err siga bajando aunque Err ya
esté subiendo: el sobreajuste es **indetectable** mirando solo err. En los datos de **validación**
ese término no existe, así que el error de validación empieza a subir cuando comienza el sobreajuste.

> *Advertencia:* el error de validación es un **estimador ruidoso**. Su mínimo no marca
> "exactamente" el punto de sobreajuste; tiene su propia varianza, que estimaremos en la Parte 2.
> Por eso se reporta $\text{Err} \pm \mathrm{SE}$ y se usan reglas como *one-standard-error*.


In [ ]:
# =============================================================
# Configuracion. Un solo generador aleatorio, con semilla fija,
# para que TODA la sesion sea reproducible.
# =============================================================
import numpy as np
import matplotlib.pyplot as plt          
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)        

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


# Parte 1 — Subajuste y sobreajuste en vivo

Toda la teoría anterior se vuelve visible con un experimento de tres líneas. La función verdadera
es $f(x) = \sin(2\pi x)$ y observamos $y_i = f(x_i) + \varepsilon_i$ con $\varepsilon_i\sim\mathcal N(0,\sigma^2)$.

Como conocemos $f$, podemos calcular **el error verdadero Err**, algo imposible con datos reales.
Esa es la ventaja de trabajar con datos sintéticos en clase.

In [ ]:
# -------------------------------------------------------------
# Datos sinteticos: conocemos la verdad f(x) = sin(2 pi x)
# -------------------------------------------------------------
def f_verdadera(x):
    return np.sin(2*np.pi*x)

SIGMA = 0.25          # desviacion estandar del ruido -> error irreducible = SIGMA**2

def genera_datos(n, sigma=SIGMA, rng=rng):
    x = np.sort(rng.uniform(0, 1, n))
    y = f_verdadera(x) + rng.normal(0, sigma, n)
    return x, y

n_train = 20
x_tr, y_tr = genera_datos(n_train)
x_te, y_te = genera_datos(500)          # conjunto de prueba grande

xx = np.linspace(0, 1, 300)
plt.plot(xx, f_verdadera(xx), "k-", lw=2, label=r"$f(x)=\sin(2\pi x)$ (verdad)")
plt.plot(x_tr, y_tr, "o", ms=7, label=f"entrenamiento (n={n_train})")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("Solo vemos los puntos; la curva negra es lo que queremos recuperar")
plt.show()

print(f"Error irreducible teorico: sigma^2 = {SIGMA**2:.4f}")

### Cuatro modelos, cuatro grados

Ajustamos polinomios de grado 1, 3, 9 y 15 con `np.polyfit`. Antes de ejecutar la celda:

> **Pregunta 1.1 .** ¿Cuál de los cuatro va a tener el **menor error de entrenamiento**?
> ¿Cuál el menor error de **prueba**? ¿Son el mismo? Escribalo antes de correr la celda.

In [ ]:
# -------------------------------------------------------------
# Ajuste polinomial para 4 grados distintos
# -------------------------------------------------------------
def ajusta_polinomio(x, y, grado):
    """Devuelve una funcion predictora ajustada por minimos cuadrados."""
    coef = np.polyfit(x, y, grado)
    return np.poly1d(coef)

def ecm(y_real, y_pred):
    return np.mean((y_real - y_pred)**2)

grados = [1, 3, 9, 15]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)

for ax, g in zip(axes.ravel(), grados):
    modelo = ajusta_polinomio(x_tr, y_tr, g)
    e_tr = ecm(y_tr, modelo(x_tr))
    e_te = ecm(y_te, modelo(x_te))
    ax.plot(xx, f_verdadera(xx), "k-", lw=2, label="verdad")
    ax.plot(xx, modelo(xx), "r-", lw=2, label=f"grado {g}")
    ax.plot(x_tr, y_tr, "o", ms=5, color="tab:blue")
    ax.set_title(f"grado {g}:  err_train={e_tr:.3f}   err_test={e_te:.3f}")
    ax.set_ylim(-2.2, 2.2)
    ax.legend(fontsize=8, loc="lower left")

fig.suptitle("Subajuste (izq. arriba) -> buen ajuste -> sobreajuste (der. abajo)", y=1.00)
fig.tight_layout()
plt.show()

> **Pregunta 1.2.** El grado 1 y el grado 15 fallan por razones **opuestas**. Describan cada
> fracaso usando el vocabulario de la Parte 0 (sesgo / varianza).
>
> **Pregunta 1.3.** El polinomio de grado 15 pasa muy cerca de casi todos los puntos azules.
> ¿Por qué eso es *malo*? ¿Qué está aprendiendo exactamente?

In [ ]:
# -------------------------------------------------------------
# LA curva: error de entrenamiento vs error de prueba
# -------------------------------------------------------------
grados_todos = np.arange(1, 16)
err_train, err_test = [], []

for g in grados_todos:
    modelo = ajusta_polinomio(x_tr, y_tr, g)
    err_train.append(ecm(y_tr, modelo(x_tr)))
    err_test.append(ecm(y_te, modelo(x_te)))

err_train, err_test = np.array(err_train), np.array(err_test)
g_opt = grados_todos[np.argmin(err_test)]

plt.semilogy(grados_todos, err_train, "o-", label=r"error de entrenamiento (err)")
plt.semilogy(grados_todos, err_test,  "s-", label=r"error de prueba ($\approx$ Err + $\sigma^2$)")
plt.axhline(SIGMA**2, color="k", ls="--", label=r"error irreducible $\sigma^2$")
plt.axvline(g_opt, color="gray", ls=":", label=f"minimo del test: grado {g_opt}")
plt.xlabel("grado del polinomio (complejidad)")
plt.ylabel("ECM  (escala log)")
plt.title("La asimetria de la Parte 0, hecha visible")
plt.legend(fontsize=9)
plt.show()

print(f"Grado optimo segun el conjunto de prueba: {g_opt}")
print(f"Numero de puntos de entrenamiento: n = {n_train}")
print(f"Grado que interpola exactamente:    {n_train-1}  (grado+1 = n parametros)")

### Preguntas sobre la curva

> **1.4.** El error de entrenamiento **nunca sube**. ¿Por qué es matemáticamente imposible que
> suba al aumentar el grado? 
>
> **1.5.** ¿En qué grado el error de entrenamiento cae prácticamente a cero? ¿Por qué **exactamente**
> ahí y no antes? Relacionen con $n=20$.
>
> **1.6.** El error de prueba nunca baja de la línea punteada, por más que ajustemos. ¿Qué es esa
> línea? ¿Qué habría que cambiar en el **experimento** — no en el modelo — para bajarla?
>
> **1.7.** ¿Es el mínimo de la curva de test un valor confiable? ¿Qué pasaría si repitiéramos todo
> con otra semilla?

### Ejercicio 1.8 — La complejidad óptima depende de $n$

Suban el tamaño de muestra de 20 a 200 sin cambiar nada más. **Predigan primero**: ¿el grado
óptimo sube, baja o se queda igual?

In [ ]:
# -------------------------------------------------------------
# Ejercicio 1.8: efecto del tamano de muestra
# -------------------------------------------------------------
plt.figure(figsize=(7.5, 5))
for n, marcador in [(20, "o-"), (50, "s-"), (200, "^-")]:
    xn, yn = genera_datos(n, rng=np.random.default_rng(SEED))   # misma semilla
    curva = [ecm(y_te, ajusta_polinomio(xn, yn, g)(x_te)) for g in grados_todos]
    g_best = grados_todos[np.argmin(curva)]
    plt.semilogy(grados_todos, curva, marcador, label=f"n={n} (optimo: grado {g_best})")

plt.axhline(SIGMA**2, color="k", ls="--", label=r"$\sigma^2$")
plt.xlabel("grado del polinomio"); plt.ylabel("error de prueba (ECM)")
plt.title("Con mas datos se puede permitir un modelo mas complejo")
plt.legend(); plt.show()

> **Discusión.** Este resultado explica por qué en un laboratorio con 12 mediciones no se entrena
> una red neuronal, y por qué las redes profundas solo despegaron cuando aparecieron conjuntos de
> datos masivos. ¿Qué implica para *su* área de investigación?

> **Observación 1.8b.** Al ejecutar la celda anterior, `numpy` emite un `RankWarning` para los
> grados altos: la matriz de diseño está **mal condicionada**. No es un capricho numérico — es la
> misma varianza alta de la que habla la teoría, manifestada como inestabilidad del álgebra lineal.
> Un modelo cuyos coeficientes no se pueden determinar con precisión es exactamente un modelo con
> varianza excesiva. En la celda siguiente silenciamos el aviso porque se repite cientos de veces,
> pero conviene verlo al menos una vez.

## 1.9 — Verificación numérica de la descomposición sesgo–varianza

La descomposición de la Parte 0 es una identidad que podemos comprobar con
código. Simulamos muchos conjuntos de entrenamiento distintos, ajustamos el mismo modelo a cada
uno, y medimos por separado el sesgo, la varianza y el ruido.

In [ ]:
# -------------------------------------------------------------
# Descomposicion empirica:  ECM = Sesgo^2 + Varianza + sigma^2
# -------------------------------------------------------------
import warnings

n_datasets = 400
x_eval = np.linspace(0.05, 0.95, 100)
f_eval = f_verdadera(x_eval)
rng_bv = np.random.default_rng(7)

filas = []
for g in [1, 3, 5, 9, 12, 15]:
    preds = np.empty((n_datasets, len(x_eval)))
    for k in range(n_datasets):
        xk, yk = genera_datos(n_train, rng=rng_bv)
        with warnings.catch_warnings():           # el RankWarning de grado alto se repite 400 veces
            warnings.simplefilter("ignore")
            preds[k] = ajusta_polinomio(xk, yk, g)(x_eval)

    pred_media = preds.mean(axis=0)
    sesgo2 = np.mean((pred_media - f_eval)**2)
    varianza = np.mean(preds.var(axis=0))
    ecm_pred = np.mean((preds - f_eval)**2)          # ECM respecto a la VERDAD
    filas.append((g, sesgo2, varianza, sesgo2 + varianza, ecm_pred))

print(f"{'grado':>6} {'Sesgo^2':>12} {'Varianza':>12} {'suma':>12} {'ECM':>12}")
print("-"*60)
for g, b, v, s, e in filas:
    print(f"{g:>6} {b:>12.3e} {v:>12.3e} {s:>12.3e} {e:>12.3e}")
print("\nNotese la escala: al pasar de grado 5 a 15 la varianza crece 12 ordenes de magnitud.")

datos = np.array(filas)
plt.plot(datos[:,0], datos[:,1], "o-", label=r"Sesgo$^2$")
plt.plot(datos[:,0], datos[:,2], "s-", label="Varianza")
plt.plot(datos[:,0], datos[:,4], "k^-", lw=2, label="ECM = suma")
plt.xlabel("grado del polinomio"); plt.ylabel("contribucion al error")
plt.yscale("log"); plt.legend()
plt.title("El compromiso sesgo-varianza, medido")
plt.show()

> **Pregunta 1.10.** Las columnas `suma` y `ECM` coinciden hasta el último decimal. ¿Por qué?
> ¿Qué habría que cambiar en el cálculo para que apareciera además el término $\sigma^2$?
> (Pista: aquí comparamos contra `f_eval`, la verdad limpia, no contra observaciones ruidosas.)
>
> **Pregunta 1.11.** ¿En qué grado se cruzan las dos curvas? ¿Coincide con el grado óptimo que
> encontró la celda de la curva de test? ¿Tiene que coincidir?

# Parte 2 — Validación cruzada

En el experimento anterior hicimos trampa: usamos 500 puntos de prueba generados a partir de la
función verdadera. **En un problema real eso no existe.** Solo tenemos los 20 puntos.

La validación cruzada resuelve exactamente eso: fabrica conjuntos de prueba a partir de los mismos
datos, rotando cuál partición se deja fuera.

In [ ]:
# -------------------------------------------------------------
# K-fold implementado a mano (para entender el Algoritmo 1)
# -------------------------------------------------------------
def k_fold_cv(x, y, grado, K=5, seed=0):
    """Devuelve los K errores de validacion, uno por particion."""
    n = len(x)
    idx = np.random.default_rng(seed).permutation(n)
    particiones = np.array_split(idx, K)          # K bloques disjuntos y exhaustivos
    errores = []
    for k in range(K):
        val_idx = particiones[k]                                    # R
        tr_idx = np.concatenate([particiones[j] for j in range(K) if j != k])  # T = D \ R
        modelo = ajusta_polinomio(x[tr_idx], y[tr_idx], grado)
        errores.append(ecm(y[val_idx], modelo(x[val_idx])))
    return np.array(errores)

e = k_fold_cv(x_tr, y_tr, grado=3, K=5, seed=0)
print("Errores por particion (grado 3, K=5):", np.round(e, 4))
print(f"Err estimado = {e.mean():.4f}  +/-  {e.std(ddof=1)/np.sqrt(len(e)):.4f}  (SE)")

In [ ]:
# -------------------------------------------------------------
# Curva de validacion cruzada: elegir el grado SIN conjunto de prueba
# -------------------------------------------------------------
grados_cv = np.arange(1, 12)
cv_media, cv_se = [], []

for g in grados_cv:
    e = k_fold_cv(x_tr, y_tr, grado=g, K=5, seed=0)
    cv_media.append(e.mean())
    cv_se.append(e.std(ddof=1)/np.sqrt(len(e)))

cv_media, cv_se = np.array(cv_media), np.array(cv_se)
g_cv = grados_cv[np.argmin(cv_media)]

plt.errorbar(grados_cv, cv_media, yerr=cv_se, fmt="o-", capsize=4,
             label="validacion cruzada K=5 (solo usa los 20 datos)")
plt.semilogy(grados_todos[:len(grados_cv)], err_test[:len(grados_cv)], "s--",
             color="gray", label="error de prueba real (500 puntos, 'hacer trampa')")
plt.yscale("log")
plt.axvline(g_cv, color="tab:red", ls=":", label=f"grado elegido por CV: {g_cv}")
plt.xlabel("grado del polinomio"); plt.ylabel("ECM")
plt.title("La CV recupera el grado optimo sin ver datos nuevos")
plt.legend(fontsize=9); plt.show()

print(f"Grado elegido por validacion cruzada : {g_cv}")
print(f"Grado optimo segun el test real      : {g_opt}")

## 2.3 — ¿Cuántas particiones? El compromiso vuelve a aparecer

In [ ]:
# -------------------------------------------------------------
# Ejercicio 2.3: K = 2, 5, 10, n (LOO)
# Repetimos TODO el experimento con 60 conjuntos de datos nuevos
# para medir el sesgo y la variabilidad del ESTIMADOR de Err.
# -------------------------------------------------------------
rng_k = np.random.default_rng(11)
n_rep = 60
datasets = [genera_datos(n_train, rng=rng_k) for _ in range(n_rep)]

# referencia: el Err verdadero, usando el conjunto de prueba grande
err_verdadero = np.mean([ecm(y_te, ajusta_polinomio(xd, yd, 3)(x_te)) for xd, yd in datasets])

print(f"Err verdadero (referencia, grado 3) = {err_verdadero:.4f}\n")
print(f"{'K':>9} {'Err medio':>11} {'sesgo':>9} {'sd entre datasets':>19}")
print("-"*52)
for K in [2, 5, 10, n_train]:
    est = np.array([k_fold_cv(xd, yd, grado=3, K=K, seed=0).mean() for xd, yd in datasets])
    etiqueta = f"{K}" + (" (LOO)" if K == n_train else "")
    print(f"{etiqueta:>9} {est.mean():>11.4f} {est.mean()-err_verdadero:>+9.4f} {est.std(ddof=1):>19.4f}")

> **Discusión 2.4.** Miren la columna `sesgo`. Con $K=2$ cada modelo entrena con solo la mitad de
> los datos, así que la CV estima el error de un modelo **peor** que el que finalmente vamos a
> entrenar: el estimador es **pesimista**. Al aumentar $K$ ese sesgo se reduce.
>
> Pero hay un precio: con $K$ grande los $K$ conjuntos de entrenamiento se parecen cada vez más
> entre sí, los $K$ errores están más correlacionados, y el promedio deja de beneficiarse de
> promediar. Es el mismo compromiso sesgo–varianza de la Parte 0, aplicado ahora al
> **procedimiento de evaluación** y no al modelo. Por eso el estándar práctico es $K=5$ o $K=10$.
>
> **2.4b.** Si repiten la CV con `seed` distinta, el resultado cambia para $K=2,5,10$ pero es
> **idéntico** para LOO. ¿Por qué LOO no depende de la semilla? ¿Es eso una ventaja?
>
> **2.5 (sin código).** ¿Es válida una CV con particiones aleatorias para:
> (a) una serie temporal de temperatura diaria? (b) tres réplicas técnicas del mismo experimento?
> (c) 20 cortes de tomografía de cada uno de 50 pacientes?
> En cada caso, ¿qué información se estaría filtrando del entrenamiento a la validación?

## 2.6 — La trampa: seleccionar variables **antes** de la validación cruzada

Éste es el error metodológico más costoso en ciencias con datos ómicos, espectroscópicos o de
imágenes, donde $p \gg n$. Vamos a caer en él a propósito.

**Situación:** 40 muestras, 2000 variables, y una etiqueta binaria generada **al azar**. No hay
ninguna señal. Cualquier modelo honesto debería acertar el 50%.

In [ ]:
# -------------------------------------------------------------
# 2.6: la trampa de la seleccion de variables (p >> n)
# -------------------------------------------------------------
rng_t = np.random.default_rng(1)
n_obs, p_var = 40, 2000
X = rng_t.normal(size=(n_obs, p_var))     # ruido puro
y_bin = rng_t.integers(0, 2, n_obs)       # etiqueta aleatoria: NO hay senal

def aciertos_cv(X_sel, y, K=5, seed=0):
    """Clasificador trivial por umbral sobre la media de las variables elegidas."""
    idx = np.random.default_rng(seed).permutation(len(y))
    parts = np.array_split(idx, K)
    acc = []
    for k in range(K):
        va = parts[k]
        tr = np.concatenate([parts[j] for j in range(K) if j != k])
        score_tr = X_sel[tr].mean(axis=1)
        umbral = np.median(score_tr)
        # aprende la orientacion en el entrenamiento
        pred_tr = (score_tr > umbral).astype(int)
        signo = 1 if (pred_tr == y[tr]).mean() >= 0.5 else 0
        pred_va = (X_sel[va].mean(axis=1) > umbral).astype(int)
        if signo == 0:
            pred_va = 1 - pred_va
        acc.append((pred_va == y[va]).mean())
    return np.mean(acc)

# --- FORMA INCORRECTA: seleccionar usando TODOS los datos, luego validar ---
corr = np.array([abs(np.corrcoef(X[:, j], y_bin)[0, 1]) for j in range(p_var)])
top = np.argsort(corr)[-10:]                       # las 10 "mejores" variables
acc_mal = aciertos_cv(X[:, top], y_bin)

# --- FORMA CORRECTA: seleccionar DENTRO de cada particion ---
def cv_correcta(X, y, K=5, seed=0):
    idx = np.random.default_rng(seed).permutation(len(y))
    parts = np.array_split(idx, K)
    acc = []
    for k in range(K):
        va = parts[k]
        tr = np.concatenate([parts[j] for j in range(K) if j != k])
        c = np.array([abs(np.corrcoef(X[tr, j], y[tr])[0, 1]) for j in range(X.shape[1])])
        top_k = np.argsort(c)[-10:]                # seleccion SOLO con el entrenamiento
        score_tr = X[np.ix_(tr, top_k)].mean(axis=1)
        umbral = np.median(score_tr)
        pred_tr = (score_tr > umbral).astype(int)
        signo = 1 if (pred_tr == y[tr]).mean() >= 0.5 else 0
        pred_va = (X[np.ix_(va, top_k)].mean(axis=1) > umbral).astype(int)
        if signo == 0:
            pred_va = 1 - pred_va
        acc.append((pred_va == y[va]).mean())
    return np.mean(acc)

acc_bien = cv_correcta(X, y_bin)

# promediamos sobre varias particiones para que el resultado no dependa de una sola
mal = np.mean([aciertos_cv(X[:, top], y_bin, seed=s) for s in range(20)])
bien = np.mean([cv_correcta(X, y_bin, seed=s) for s in range(20)])

print("Datos: 40 muestras, 2000 variables de RUIDO PURO, etiqueta ALEATORIA")
print(f"  Seleccion ANTES de la CV  (INCORRECTO): {mal:6.1%}")
print(f"  Seleccion DENTRO de la CV (CORRECTO)  : {bien:6.1%}")
print(f"  Valor honesto esperado                : {0.5:6.1%}")
print("\nEl 'exito' del primero es 100% ilusorio: no hay nada que aprender en estos datos.")

> **Pregunta 2.7.** ¿De dónde salió la señal si los datos son ruido puro? Expliquen por qué el
> paso de selección, hecho una sola vez sobre todos los datos, **filtra** información de las
> etiquetas de validación hacia el modelo.
>
> **Regla general:** *todo* paso que mire las etiquetas — selección de variables, escalado,
> imputación, ajuste de hiperparámetros — debe ocurrir **dentro** de cada pliegue. En `sklearn`
> esto es exactamente lo que garantiza un `Pipeline`.
>
> **2.8.** Suban `p_var` de 2000 a 20000. ¿El sesgo optimista empeora? ¿Por qué?

# Parte 3 — Población, muestra y bootstrap

## Definiciones

- **Población:** el conjunto completo de objetos o individuos sobre el que queremos concluir.
- **Muestra:** un subconjunto obtenido de la población, idealmente al azar.

## Tipos de muestreo

1. **Con reemplazo** — cada elemento puede salir repetido. Genera observaciones i.i.d.
2. **Sin reemplazo** — cada elemento sale a lo más una vez.
3. **Aleatorio simple** — todas las muestras del mismo tamaño son igualmente probables.
4. **Estratificado** — se muestrea dentro de subgrupos definidos de antemano.
5. **Sesgado** — el mecanismo de selección depende de lo que se quiere medir; **no** representa a la población.

## El problema central de la inferencia

Tenemos **una sola** muestra y calculamos un estadístico $\hat\theta$ (media, mediana, pendiente,
$R^2$...). ¿Cuánto vale $\theta$ en la población, y con qué incertidumbre?

Para la media existe una fórmula: $\mathrm{SE}(\bar X) = \sigma/\sqrt{n}$ (Sesión 02a).
**¿Y para la mediana? ¿Para el coeficiente de correlación? ¿Para el pico de un espectro?**
Ahí las fórmulas se acaban — o son horribles. El bootstrap las reemplaza por simulación.

In [ ]:
# -------------------------------------------------------------
# Tres poblaciones: longitud de aleta de tres tipos de pez
# -------------------------------------------------------------
def grafica_pdfs(x, distribuciones):
    """Grafica un diccionario {etiqueta: (mu, sigma)} de normales."""
    fig, ax = plt.subplots()
    f = {}
    for etiqueta, (mu, sigma) in distribuciones.items():
        f[etiqueta] = stats.norm(loc=mu, scale=sigma)
        ax.plot(x, f[etiqueta].pdf(x), lw=3, alpha=0.7,
                label=f"{etiqueta}: $\\mu$={mu}, $\\sigma$={sigma}")
    ax.set_xlabel("longitud de la aleta [cm]")
    ax.set_ylabel("densidad de probabilidad")
    ax.legend()
    return f

params = {"tipo 1": (12, 2), "tipo 2": (16, 3), "tipo 3": (18, 1)}
x = np.linspace(0, 30, 300)
f = grafica_pdfs(x, params)
plt.title("Poblaciones VERDADERAS (en la practica, desconocidas)")
plt.show()

In [ ]:
# -------------------------------------------------------------
# Lo unico que un biologo observa realmente: 20 peces de cada tipo
# -------------------------------------------------------------
n_peces = 20
muestras = {k: f[k].rvs(n_peces, random_state=rng) for k in params}   # reproducible

bins = np.linspace(6, 24, 25)
for k, d in muestras.items():
    plt.hist(d, bins=bins, density=True, alpha=0.5, label=k)          # alpha: no se tapan
for k, (mu, s) in params.items():
    plt.plot(x, stats.norm(mu, s).pdf(x), lw=1.5, ls="--")
plt.xlim(6, 24); plt.xlabel("longitud de la aleta [cm]"); plt.ylabel("densidad")
plt.title(f"Muestras de n={n_peces} (barras) vs poblaciones verdaderas (lineas)")
plt.legend(); plt.show()

for k, d in muestras.items():
    print(f"{k}: media muestral = {d.mean():6.2f}   (verdadera = {params[k][0]})")

> **Pregunta 3.1.** Los histogramas con $n=20$ se parecen poco a las curvas punteadas.
> ¿Es eso un problema de los datos, del método, o es lo esperado? ¿Cómo cuantificamos "poco"?

## Bootstrapping

**Idea:** si no podemos repetir el experimento 1000 veces, **remuestreamos con reposición** la
muestra que sí tenemos, tratándola como si fuera la población.

Dada una muestra $x = (x_1,\dots,x_n)$, generamos $B$ remuestreos de tamaño $n$ **con reposición**:

$$x^{*(1)} = (x_1, x_1, x_2, x_1, x_5,\dots), \quad
x^{*(2)} = (x_2, x_4, x_2, x_1, x_5,\dots), \quad \dots$$

Para cada uno calculamos el estadístico de interés, obteniendo
$\hat\theta^{*(1)},\hat\theta^{*(2)},\dots,\hat\theta^{*(B)}$.

La **distribución bootstrap** de esos $B$ valores aproxima la distribución muestral de $\hat\theta$.
De ahí salen:

$$\widehat{\mathrm{SE}}(\hat\theta) = \mathrm{sd}\!\left(\hat\theta^{*(1)},\dots,\hat\theta^{*(B)}\right)
\qquad
\text{IC}_{95\%} = \left[\,q_{2.5}(\hat\theta^*),\; q_{97.5}(\hat\theta^*)\,\right]$$

> ⚠️ **El error más común.** El error estándar es la desviación **entre** remuestreos,
> `np.std(medias_bootstrap)`. **No** es el promedio de las desviaciones *dentro* de cada
> remuestreo, `np.mean([np.std(b) for b in remuestreos])` — esa cantidad estima $\sigma$
> poblacional, y es $\sqrt{n}$ veces más grande. Confundirlas infla la barra de error por un
> factor 4.5 cuando $n=20$.

In [ ]:
# -------------------------------------------------------------
# Bootstrap, vectorizado y con el estadistico como parametro
# -------------------------------------------------------------
def bootstrap(datos, estadistico=np.mean, B=5000, rng=rng):
    """
    Remuestrea `datos` CON REPOSICION B veces y aplica `estadistico` a cada remuestreo.

    Devuelve un arreglo de B valores: la distribucion bootstrap del estadistico.
    """
    datos = np.asarray(datos)
    n = len(datos)
    idx = rng.integers(0, n, size=(B, n))        # con reposicion: indices repetidos permitidos
    return estadistico(datos[idx], axis=1)

def resumen_bootstrap(datos, estadistico=np.mean, B=5000, nivel=95, rng=rng):
    theta_star = bootstrap(datos, estadistico, B, rng)
    alfa = (100 - nivel) / 2
    return {
        "estimacion": estadistico(np.asarray(datos)),
        "SE": theta_star.std(ddof=1),                       # <- dispersion ENTRE remuestreos
        "IC": np.percentile(theta_star, [alfa, 100 - alfa]),
        "distribucion": theta_star,
    }

# demostracion del error clasico
d1 = muestras["tipo 1"]
b = bootstrap(d1, np.mean, B=5000)
sd_dentro = bootstrap(d1, np.std, B=5000).mean()

print(f"n = {len(d1)}")
print(f"  sd ENTRE remuestreos  = {b.std(ddof=1):.3f}   <- SE de la media (CORRECTO)")
print(f"  sd DENTRO (promedio)  = {sd_dentro:.3f}   <- estima sigma poblacional (= {params['tipo 1'][1]})")
print(f"  formula teorica sigma/sqrt(n) = {params['tipo 1'][1]/np.sqrt(len(d1)):.3f}")
print(f"  cociente entre ambas  = {sd_dentro/b.std(ddof=1):.2f}   ~  sqrt(n) = {np.sqrt(len(d1)):.2f}")

In [ ]:
# -------------------------------------------------------------
# Resultado final para los tres tipos de pez
# -------------------------------------------------------------
print(f"{'':>8} {'media':>8} {'SE':>7} {'IC 95%':>18} {'mu real':>9} {'dentro IC?':>11}")
print("-"*68)
for k, d in muestras.items():
    r = resumen_bootstrap(d, np.mean, B=5000)
    mu_real = params[k][0]
    dentro = r["IC"][0] <= mu_real <= r["IC"][1]
    print(f"{k:>8} {r['estimacion']:>8.2f} {r['SE']:>7.3f} "
          f"[{r['IC'][0]:6.2f}, {r['IC'][1]:6.2f}] {mu_real:>9} {str(dentro):>11}")

> **Pregunta 3.2.** El IC del 95% se llama así porque, **repitiendo el experimento** muchas veces,
> el 95% de los intervalos construidos contendría el valor verdadero. Con tres tipos de pez,
> ¿cuántos esperarían que fallen? ¿Qué NO significa "95% de probabilidad de que $\mu$ esté aquí"?
>
> **Ejercicio 3.3.** Repitan el ejercicio con 100 muestras distintas y cuenten qué fracción de los
> IC contiene el $\mu$ verdadero. ¿Se acerca a 0.95?

## 3.4 — ¿Por qué funciona? Bootstrap vs. la distribución muestral real

Aquí sí podemos hacer trampa: como conocemos la población, podemos generar la distribución muestral
**de verdad**, repitiendo el experimento 2000 veces. El bootstrap solo tuvo acceso a 20 números.

> ⚠️ Comparar es válido **únicamente si ambos usan el mismo $n$**. En la versión original del
> notebook se comparaban medias de $n=100$ contra bootstrap de $n=20$: los anchos difieren por
> $\sqrt{5}\approx 2.24$ y la comparación no dice nada.

In [ ]:
# -------------------------------------------------------------
# 3.4: la distribucion muestral REAL vs la bootstrap. MISMO n.
# -------------------------------------------------------------
B = 2000
n = len(d1)                                    # <-- mismo n en ambos casos

# (a) Imposible en la practica: repetir el experimento B veces
medias_reales = np.array([f["tipo 1"].rvs(n, random_state=rng).mean() for _ in range(B)])

# (b) Lo que si podemos hacer: bootstrap sobre los 20 datos que tenemos
medias_boot = bootstrap(d1, np.mean, B=B)

rejilla = np.linspace(9.5, 14.5, 60)
plt.hist(medias_reales, bins=rejilla, density=True, alpha=0.55,
         label=f"distribucion muestral REAL (repetir el experimento {B} veces)")
plt.hist(medias_boot, bins=rejilla, density=True, alpha=0.55,
         label=f"distribucion BOOTSTRAP (solo n={n} datos)")
plt.axvline(params["tipo 1"][0], color="k", ls="--", label=r"$\mu$ verdadero")
plt.axvline(d1.mean(), color="tab:red", ls=":", label=r"$\bar{x}$ observado")
plt.xlabel("media de la longitud de aleta [cm]"); plt.ylabel("densidad")
plt.title("El bootstrap reconstruye la distribucion muestral sin acceso a la poblacion")
plt.legend(fontsize=8); plt.show()

print(f"sd de la distribucion muestral real : {medias_reales.std(ddof=1):.3f}")
print(f"sd de la distribucion bootstrap     : {medias_boot.std(ddof=1):.3f}")
print(f"prediccion teorica sigma/sqrt(n)    : {params['tipo 1'][1]/np.sqrt(n):.3f}")

> **Pregunta 3.5.** Los dos histogramas tienen **anchos** casi idénticos pero están **centrados**
> en puntos distintos. ¿Por qué? ¿Cuál de los dos centros es el correcto y cuál es el que el
> bootstrap puede conocer?
>
> **Pregunta 3.6.** ¿Cuál de las dos distribuciones podrían calcular en un experimento real?
> ¿De dónde saca el bootstrap la información que parece no tener?
>
> **Pregunta 3.7.** Bajen `n` a 5 y repitan. ¿Sigue funcionando? ¿A partir de qué $n$ dejarían de
> confiar en el resultado?
>
> **Conexión con la Sesión 02a.** El ancho coincide con $\sigma/\sqrt{n}$, el mismo resultado que
> obtuvimos allá con las flores rosadas — pero aquí **sin usar ninguna fórmula**, solo remuestreando.

# Parte 4 — Ejercicios propuestos

## 4.1 — Medir el término de complejidad de SURE

La Parte 0 afirma que $\sum_i \partial\hat f_i/\partial y_i = p$ (número de parámetros) para un
ajuste lineal. Verifíquenlo numéricamente: perturben **un solo** $y_i$ en $+\delta$, reajusten, y
midan cuánto se movió $\hat f_i$.

In [ ]:
# -------------------------------------------------------------
# 4.1 SOLUCION: el termino de complejidad = numero de parametros
# -------------------------------------------------------------
delta = 1e-4
print(f"{'grado':>6} {'parametros (p)':>16} {'suma d(f_i)/d(y_i)':>22}")
print("-"*46)
for g in [1, 2, 3, 5, 9]:
    base = ajusta_polinomio(x_tr, y_tr, g)(x_tr)
    traza = 0.0
    for i in range(len(y_tr)):
        y_pert = y_tr.copy()
        y_pert[i] += delta
        pert = ajusta_polinomio(x_tr, y_pert, g)(x_tr)
        traza += (pert[i] - base[i]) / delta       # derivada numerica
    print(f"{g:>6} {g+1:>16} {traza:>22.4f}")

> **Pregunta.** ¿Por qué esta suma se llama *grados de libertad efectivos*? ¿Qué pasaría con una
> regresión **regularizada** (ridge)? ¿Sería mayor o menor que $p$?

## 4.2 — Romper el bootstrap a propósito

Apliquen exactamente el mismo procedimiento al **máximo** de la muestra en vez de a la media.

In [ ]:
# -------------------------------------------------------------
# 4.2: el bootstrap FALLA para estadisticos de extremos
# -------------------------------------------------------------
maximos = bootstrap(d1, np.max, B=5000)

plt.hist(maximos, bins=40, density=True, color="tab:orange")
plt.axvline(d1.max(), color="k", ls="--", label=f"maximo observado = {d1.max():.2f}")
plt.xlabel("maximo bootstrap"); plt.ylabel("densidad")
plt.title("Distribucion bootstrap del maximo: un peine, no una campana")
plt.legend(); plt.show()

print(f"valores distintos que toma el maximo bootstrap: {len(np.unique(maximos))}")
print(f"fraccion de remuestreos que alcanzan el maximo observado: "
      f"{np.mean(maximos == d1.max()):.3f}   (teorico: 1-(1-1/n)^n = {1-(1-1/len(d1))**len(d1):.3f})")
print(f"maximo bootstrap NUNCA supera el observado: {maximos.max() == d1.max()}")

> **Preguntas 4.2.** (a) ¿Por qué el histograma es un peine con pocos valores distintos?
> (b) ¿Por qué el máximo bootstrap **nunca** supera al máximo observado, mientras que el máximo
> de una muestra real sí podría? (c) ¿Qué dice esto sobre la afirmación "el bootstrap sirve para
> cualquier estadístico"? (d) ¿En qué problemas de física o ingeniería importa justamente la cola
> de la distribución?

## 4.3 — Bootstrap sobre un ajuste: la pendiente con barra de error

Retomen los datos $(x,y)$ de la Sesión 02a. Remuestreen **los pares** $(x_i, y_i)$ — no las $x$ y
las $y$ por separado — y obtengan la distribución de la pendiente.

In [ ]:
# -------------------------------------------------------------
# 4.3 SOLUCION: incertidumbre de la pendiente por bootstrap de pares
# -------------------------------------------------------------
rng_r = np.random.default_rng(3)
n_pts = 40
x_reg = rng_r.uniform(0, 10, n_pts)
y_reg = 2.0*x_reg + 1.0 + rng_r.normal(0, 2.0, n_pts)    # pendiente verdadera = 2.0

def pendiente_boot(x, y, B=5000, rng=rng_r):
    n = len(x)
    idx = rng.integers(0, n, size=(B, n))                 # se remuestrean los PARES
    return np.array([np.polyfit(x[i], y[i], 1)[0] for i in idx])

pend = pendiente_boot(x_reg, y_reg)
b_obs = np.polyfit(x_reg, y_reg, 1)[0]
ic = np.percentile(pend, [2.5, 97.5])

print(f"pendiente ajustada  = {b_obs:.3f}")
print(f"SE bootstrap        = {pend.std(ddof=1):.3f}")
print(f"IC 95%              = [{ic[0]:.3f}, {ic[1]:.3f}]")
print(f"pendiente verdadera = 2.000   -> dentro del IC: {ic[0] <= 2.0 <= ic[1]}")

plt.hist(pend, bins=40, density=True, alpha=0.7)
plt.axvline(2.0, color="k", ls="--", label="valor verdadero")
plt.axvline(b_obs, color="tab:red", ls=":", label="estimacion")
plt.xlabel("pendiente"); plt.ylabel("densidad")
plt.title("La recta de la Sesion 02a, ahora con barra de error")
plt.legend(); plt.show()

> **Pregunta.** Comparen con el error estándar analítico
> $\mathrm{SE}(\hat\beta) = \hat\sigma/\sqrt{\sum(x_i-\bar x)^2}$. ¿Coinciden? ¿En qué situación
> el bootstrap daría un resultado **mejor** que la fórmula? (Pista: ¿qué supone la fórmula sobre
> el ruido?)

## 4.4 — Protocolo completo (pregunta de cierre, sin código)

Tienen **40 mediciones** de un experimento y quieren decidir entre un modelo lineal, uno cuadrático
y una red neuronal pequeña. Describan en cinco líneas el protocolo completo:

1. ¿Cómo particionan los datos?
2. ¿Qué se decide con validación cruzada y qué no?
3. ¿Qué número reportan en el artículo?
4. **¿Por qué el número que publican NO puede ser el mejor error de validación que vieron?**
5. ¿Cómo reportan la incertidumbre de ese número?

